# FND-1 Module 05: Descriptive results

Trace each synthetic-cohort summary to its source, denominator, time window, and interpretation limit. No real clinical or population inference is supported.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
DATA, OUTPUTS = ROOT / 'data', ROOT / 'outputs'
source = pd.read_csv(DATA / 'resolved-analytic-table.csv', keep_default_na=False)
profiles = pd.read_csv(OUTPUTS / 'variable-profile.csv', keep_default_na=False)
registry = pd.read_csv(OUTPUTS / 'denominator-registry.csv', keep_default_na=False)
source.shape, profiles.shape, registry.shape

## 1. Verify source and grain

The accepted source must remain 374 rows, 374 people, and 29 fields before any summary is interpreted.

In [ ]:
digest = hashlib.sha256((DATA / 'resolved-analytic-table.csv').read_bytes()).hexdigest()
assert digest == '3c9944edc3806aa3b709a9ca08a9986a2f79978b1074ed098e31f19b533db25a'
assert source.shape == (374, 29)
assert source.patient_id.nunique() == source.index_encounter_id.nunique() == 374
{'rows': len(source), 'people': source.patient_id.nunique(), 'fields': len(source.columns)}

## 2. Compare summaries and denominators

Means and medians answer different descriptive questions. Available-case timing uses 111 recorded next encounters; structural blanks are not zeros.

In [ ]:
selected = profiles.loc[profiles.result_id.isin(['VP02', 'VP09', 'VP14']), ['result_id', 'available_n', 'missing_n', 'mean', 'median', 'q1', 'q3', 'maximum', 'retained_conditions']]
assert int(selected.loc[selected.result_id == 'VP14', 'available_n'].iloc[0]) == 111
selected

## 3. Reconcile cross-tabs, rates, and strata

Complete cells, exact denominators, and unadjusted labels keep descriptive tables from implying unsupported comparisons.

In [ ]:
cross_tabs = pd.read_csv(OUTPUTS / 'cross-tabs.csv', keep_default_na=False)
rates = pd.read_csv(OUTPUTS / 'rates.csv', keep_default_na=False)
strata = pd.read_csv(OUTPUTS / 'stratified-table.csv', keep_default_na=False)
checks = pd.read_csv(OUTPUTS / 'descriptive-checks.csv', keep_default_na=False)
assert cross_tabs.groupby('result_id').n.sum().to_dict() == {'CT01': 374, 'CT02': 374}
assert rates.set_index('result_id').loc['RT05', 'numerator'] == 36
assert strata.n.sum() == 374 and (checks.status == 'pass').all()
rates[['result_id', 'numerator', 'denominator', 'percent', 'wilson_95_lower_percent', 'wilson_95_upper_percent']]

## 4. Module 06 handoff decision

Reference disposition: **accept with conditions**. Module 06 must use exact released rows, retain N01 through N08, keep small internal cells reviewable, label strata unadjusted, and preserve the synthetic-data claim boundary.